# DiagnosticIQ source step 5: asset descriptions and equipment profiles

Reads the source CSVs in `$ASSETOPSBENCH_DATASET_DIR/data/` and writes `extracted/equipment_info/`: `gre_desc.json` (the asset descriptions used in every question), `rule_to_equipment_map.json` and `rules_without_equipment_info.txt`. Run it after step 3, which adds the `asset` field.

In [ ]:
import os
import re
import json
from datetime import datetime
from typing import Optional, List, Callable, Any, Union, Dict, Hashable
from abc import abstractmethod, ABC
import numpy as np
import math
import pandas as pd
import copy

In [ ]:
#parameter 

from dataset_utils.paths import get_dataset_dir

dataset_dir = get_dataset_dir()
extract_fol = dataset_dir / "extracted"
data_fol = dataset_dir / "data"
dataset_loc = os.path.join(extract_fol,"observations.rules.json")
alert_event_loc = os.path.join(data_fol, "alert_events.csv")
equipment_profile_loc = os.path.join(data_fol, "equipment_profile.csv")
save_equipment_info_fol = extract_fol / "equipment_info"

os.makedirs(save_equipment_info_fol, exist_ok=True)

In [ ]:
with open(dataset_loc, "r") as f0:
    data = json.load(f0)

def text_write(text, filename= "debug.test"):
    with open(filename, "w") as f0:
        f0.write(text)

In [ ]:
asset_list = {}
rule_ids = set()
for d in data["rule_set"]:
    asset = d["asset"]
    rule_ids.add(d["id"].strip())
    if asset is None:
        asset = "None"
    try:
        asset_list[asset] += 1
    except KeyError:
        asset_list[asset] = 1
        
stats = pd.DataFrame.from_dict(asset_list, orient="index").reset_index()
stats.columns = columns=["asset","#instances"]
stats.sort_values("#instances", ascending=False)

In [ ]:
alert_events = pd.read_csv(alert_event_loc)
equipment_profile = pd.read_csv(equipment_profile_loc)

In [ ]:
#only several rules have equipment mapping.
print("Intersection of rules:{}".format(set(alert_events["rule_id"].to_list()).intersection(rule_ids)))
print("Num of rules without equipments:{}".format(len(rule_ids-set(alert_events["rule_id"].to_list()))))


text_write("\n".join(rule_ids-set(alert_events["rule_id"].to_list())), os.path.join(save_equipment_info_fol, "rules_without_equipment_info.txt"))

In [ ]:
count_rule_applied_equipment = alert_events[
    ["equipment_id", "equipment_name", "rule_id"]
    ].groupby("rule_id").count().sort_values(["equipment_id"], ascending=False)

count_rule_applied_equipment


In [ ]:
equipment_info_to_retrieve = pd.DataFrame()#index=count_rule_applied_equipment.index
rule_id_group = alert_events[["equipment_id", "equipment_name", "rule_id"]].groupby("rule_id")
equipment_info_to_retrieve["rule_id"] = rule_id_group["rule_id"].transform(lambda x: "||".join(set(x.to_list())))
equipment_info_to_retrieve["equipment_ids"] = rule_id_group["equipment_id"].transform(lambda x: "||".join(set(x.to_list())))
equipment_info_to_retrieve["equipment_names"] = rule_id_group["equipment_name"].transform(lambda x: "||".join(set(x.to_list())))
equipment_info_to_retrieve.drop_duplicates(inplace=True)


In [ ]:
equipment_info_to_retrieve

In [ ]:
equipment_info_of_rule_ids = {}
for _, rule in equipment_info_to_retrieve.to_dict('index').items():
    _temp = {}
    equipments_per_rule = equipment_profile[equipment_profile["equipment_id"].isin(rule["equipment_ids"].split("||"))]
    
    _temp["rule_id"] = rule["rule_id"]
    _temp["equipment_profiles"] = [v for _,v in equipments_per_rule.to_dict('index').items()]

    print("Rule: {} | #equipment_profiles: {}".format(
        _temp ["rule_id"],
        len(_temp["equipment_profiles"])
        ))
    
    equipment_info_of_rule_ids[rule["rule_id"]] = _temp

In [ ]:
def replace_list(d:Dict[str,Any]) -> Dict[str,Any]:
    _temp = {}
    for k,v in d.items():
        if isinstance(v, dict):
            _temp[k] = replace_list(v)
        else:
            _temp[k] = []

    return _temp

def remove_nan_in_list(d:Dict[str,Any]) -> Dict[str,Any]:
    _temp = copy.deepcopy(d)
    for k,v in d.items():
        if isinstance(v, dict):
            _temp[k] = remove_nan_in_list(v)
        else:
            #print(k,_temp[k])
            #print([str(x) for x in _temp[k]])
            _temp[k] = [x for x in _temp[k] if not str(x) in ['NaN', 'nan']]

    return _temp

def unique_elements(d:Dict[str,Any]) -> Dict[str,Any]:
    _temp = copy.deepcopy(d)
    for k,v in d.items():
        if isinstance(v, dict):
            _temp[k] = remove_nan_in_list(v)
        else:
            _temp[k] = list(set(_temp[k]))

    return _temp

def concat_dict(d:List[Dict[str,Any]], template:Dict[str,Any]) -> None:

    for k,v in template.items():
        if isinstance(v, dict):
            #print(k,[x[k] for x in d], template[k] )
            concat_dict([x[k] for x in d], template[k])
        else:
            #print(k,d)
            template[k] = [x[k] for x in d]
            
def save_json(data:Dict[Hashable,Any], loc:str) -> None:
    with open(loc, "w") as f0:
        json.dump(data, f0)

def aggregate_profile(profiles:List[dict]) -> dict:
    
    if len(profiles)<1:
        return {}
    
    if len(profiles)<2:
        return profiles[0]

    _temp = copy.deepcopy(profiles[0])
    _temp = replace_list(_temp)

    #aggregate values to list
    concat_dict(profiles,_temp)
    _temp = remove_nan_in_list(_temp)
    _temp = unique_elements(_temp)
    

    return _temp


In [ ]:
rule_to_equipment_profile = {}
for k,v in equipment_info_of_rule_ids.items():
    check = aggregate_profile(v['equipment_profiles'])
    rule_to_equipment_profile[k] = check

save_json(rule_to_equipment_profile, os.path.join(save_equipment_info_fol,"rule_to_equipment_map.json"))



In [ ]:
gre_asset_description = pd.read_csv(os.path.join(data_fol, "GRE_Asset_Description.csv"))
gre_asset_description.set_index(['Asset Type Name'], inplace=True)

gre_asset_description.columns = ['desc', 'detailed_desc']
gre_asset_description = gre_asset_description.to_dict(orient="index")

In [ ]:
save_json(gre_asset_description, os.path.join(save_equipment_info_fol, "gre_desc.json"))